In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [2]:
data = pd.read_csv("/kaggle/input/competitions/digit-recognizer/train.csv")
data = np.array(data) #converting pandadataframes to array to compute
r, c = data.shape
np.random.shuffle(data) #shuffeling the data set to prevent overfitting

In [3]:
data_ = data[0:1000] #developemnt and validation set
data_ = data_.T #transpose the matrix (m.n - n.m)
print(data_)
Y_dev = data_[0] # getting lables
#X_dev = data_[1:c] #taking activation of all the pixels 
X_dev = data_[1:c] / 255.0

[[8 6 2 ... 2 4 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]]


In [4]:
data_Train = data[1000:r].T 
Y_Train = data_Train[0] #getting lables
#X_Train = data_Train[1:c] #taking actiavtion of all pixels
X_Train = data_Train[1:c] / 255.0

In [5]:
def init_Para():
    #for 1st layer
    W1 = np.random.randn(10, 784) * 0.01 #10 rows and 784 col having 784 weights for each rows (neurons) i.e 10
    b1 = np.zeros((10, 1)) #10 rows with 1 col having 1 bias for each rows (neurons)
    #for 2nd layer 
    W2 = np.random.randn(10, 10) * 0.01
    b2 = np.zeros((10, 1))
    return W1,b1,W2,b2

In [6]:
def sigmoid(Zi): #activation function
    #return 1 / (1 + np.exp(-Zi))
    """Forward pass: ReLU activation"""
    return np.maximum(0, Zi)

In [7]:
def derv_sigmoid(Z):
    
    return (Z > 0).astype(float)
    #s = sigmoid(Z)
   #return s * (1 - s)

In [8]:
def softmax(Zi):
    exp_Z = np.exp(Zi - np.max(Zi, axis=0, keepdims=True)) # Max subtraction prevents overflow!
    return exp_Z / np.sum(exp_Z, axis=0, keepdims=True)

In [9]:
def forward_pass(W1,b1,W2,b2,X):
    Z1 = W1.dot(X) + b1
    A1 = sigmoid(Z1)
    Z2 = W2.dot(A1) + b2
    A2 = softmax(Z2)
    return Z1,A1,Z2,A2

In [10]:
print(Y_Train.size)


41000


In [11]:
def onehot(number_of_lables,number_of_neurons=10):
    n = number_of_lables.shape[0]
    Output_matrix = np.zeros((n, number_of_neurons))
    Output_matrix = Output_matrix
    for x,i in enumerate(number_of_lables):
        Output_matrix[x][i] = 1
    return Output_matrix.astype(int).T
        
    

In [12]:
a = Y_Train
onehot(a)

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 1],
       [1, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 1, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(10, 41000))

In [13]:
def Backprop_single(Z1,A1,Z2,A2,W2,X, Y):
    m = Y.size
    one_hotY = onehot(Y)
    dz2 = A2 - one_hotY
    dw2 = 1 / m * dz2.dot(A1.T)
    db2 = 1 / m * np.sum(dz2, axis=1, keepdims=True)
    dz1 = W2.T.dot(dz2) * derv_sigmoid(Z1)
    dw1 = 1 / m * dz1.dot(X.T)
    db1 = 1 / m * np.sum(dz1, axis=1, keepdims=True)

    return dw1,db1,dw2,db2
    



# for single image SGD
    
#     loss = (Y-A2) ** 2 #correct out put - predecited for each neuron
# #-----FOR OUTPUT LAYER--------
#     #Chain rule
#     dC_da2 = 2*(A2-Y) #d of loss wrt to previous activation
#     da_dz2 = A2 * (1 - A2) #d of loss wrt to prvious weightsum
#     dz_dw2 = A1 #d of loss wrt to previous weight
#     dc_dw2 = dc_da2 * da_dz2 * dz_dw #gradient of w2 #change in loss(cost) of each neuron wrt to its change it weight
#     dc_dw2 = dc_da2 * da_dz2 * 1 #gradient of w2 #change in loss(cost) of each neuron wrt to its change it bias

# #passing error in hidden layer
#     dC_da1 = dc_da2 * da_dz2 * w2

    
# #-----FOR HIDDEN LAYER-------
#     da_dz1 = A1 * (1 - A1) #d of loss wrt to prvious weightsum
#     dz_dw1 = X #d of loss wrt to previous weight
#     dc_dw1 = dc_da * da_dz1 * dz_dw #gradient of w1 #change in loss(cost) of each neuron wrt to its change it weight
#     dc_dw1 = dc_da * da_dz1 * 1  #gradient of w1 #change in loss(cost) of each neuron wrt to its change it bias
   

In [14]:
def Updating_Param(w1,b1,w2,b2,dw1,db1,dw2,db2,omega):
    w1 = w1 - omega * dw1
    b1 = b1 - omega * db1
    w2 = w2 - omega * dw2
    b2 = b2 - omega * db2
    return w1,b1,w2,b2

In [15]:
def get_predections(A2):
    return np.argmax(A2, 0)

In [16]:
def get_accuracy(predections,Y):
    print(predections, Y)
    return np.sum(predections == Y) / Y.size

In [17]:
def Batch_GD(X,Y,itter,omega):
    w1,b1,w2,b2 = init_Para()
    for i in range(itter):
        z1,a1,z2,a2 = forward_pass(w1,b1,w2,b2,X)
        dw1, db1, dw2, db2 = Backprop_single(z1, a1, z2, a2, w2, X, Y)
        w1,b1,w2,b2 = Updating_Param(w1,b1,w2,b2,dw1,db1,dw2,db2,omega)
        if i % 50 == 0:
            print("Total itterations :", i)
            print("Accuraccy: ", get_accuracy(get_predections(a2),Y))
    return w1,b1,w2,b2

In [18]:
w1 ,b1 ,w2 ,b2 = Batch_GD(X_Train,Y_Train,1000,0.1)

Total itterations : 0
[5 5 6 ... 4 6 9] [2 3 6 ... 7 6 1]
Accuraccy:  0.09385365853658537
Total itterations : 50
[3 3 7 ... 7 0 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.321
Total itterations : 100
[3 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.5885121951219512
Total itterations : 150
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.7438536585365854
Total itterations : 200
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.8159512195121951
Total itterations : 250
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.8462439024390244
Total itterations : 300
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.8656341463414634
Total itterations : 350
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.878
Total itterations : 400
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.886390243902439
Total itterations : 450
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.8920731707317073
Total itterations : 500
[2 3 6 ... 7 6 1] [2 3 6 ... 7 6 1]
Accuraccy:  0.8959756097560976
Total itterations : 550
[2 3